# 정류장 위·경도 매핑

국토교통부 전국 버스정류장 위치정보를 정류장명과 시도코드 기준으로 매칭한다.

### 셀 1. 기존 노선 정류장명에 국토부 위·경도 매핑\n

### ?? ? 1. ??? ??? ???? ?? ? ?? ?? ?? ??


In [ ]:
from pathlib import Path
import re
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'gtx_a_seoul_bus_outputs' / 'transport_card'
STOP_DIR = DATA_DIR / 'route_stops'
LOCATION_PATH = BASE_DIR / '국토교통부_전국 버스정류장 위치정보_20251031.csv'
DATES = ['20241017', '20251016']

def normalize_name(value):
    if pd.isna(value): return ''
    value = re.sub(r'\s+', '', str(value).strip())
    value = re.sub(r'\((?:중|미정차|경유)\)', '', value)
    return re.sub(r'[.·ㆍ/\-]', '', value)

location = pd.read_csv(LOCATION_PATH, dtype=str, encoding='cp949').fillna('')
location['위도'] = pd.to_numeric(location['위도'], errors='coerce')
location['경도'] = pd.to_numeric(location['경도'], errors='coerce')
location['_name_key'] = location['정류장명'].map(normalize_name)
location = location.dropna(subset=['위도', '경도'])
coord_lookup = {}
for ctpv_cd, region_name in [('41', '경기'), ('11', '서울')]:
    region = location[location['도시명'].str.contains(region_name, na=False)]
    coord_lookup[ctpv_cd] = region.drop_duplicates('_name_key').set_index('_name_key')[['위도', '경도']].to_dict('index')

def coordinate_by_name(name, ctpv_cd):
    key = normalize_name(name)
    ctpv_cd = str(ctpv_cd).strip()
    item = coord_lookup.get(ctpv_cd, {}).get(key)
    if item is None: return pd.Series({'위도': pd.NA, '경도': pd.NA})
    return pd.Series(item)

for date in DATES:
    raw = pd.read_csv(DATA_DIR / f'gtx_a_transport_card_{date}_raw.csv', dtype=str, encoding='utf-8-sig').fillna('')
    stops = pd.read_csv(STOP_DIR / f'gtx_a_route_stops_{date}.csv', dtype=str, encoding='utf-8-sig').fillna('')
    stops['query_route_id'] = stops['query_route_id'].str.strip(); stops['sttnId'] = stops['sttnId'].str.strip()
    ref = stops[['query_route_id', 'sttnId', 'sttnNm', 'sttnSeq']].drop_duplicates(['query_route_id', 'sttnId'])
    ride_ref = ref.rename(columns={'sttnId':'ride_sttn_id','sttnNm':'승차정류장명','sttnSeq':'승차정류장순서'})
    goff_ref = ref.rename(columns={'sttnId':'goff_sttn_id','sttnNm':'하차정류장명','sttnSeq':'하차정류장순서'})
    result = raw.merge(ride_ref, on=['query_route_id','ride_sttn_id'], how='left').merge(goff_ref, on=['query_route_id','goff_sttn_id'], how='left')
    ride = result.apply(lambda r: coordinate_by_name(r['승차정류장명'], r.get('ride_ctpv_cd','41')), axis=1).rename(columns={'위도':'승차위도','경도':'승차경도'})
    goff = result.apply(lambda r: coordinate_by_name(r['하차정류장명'], r.get('goff_ctpv_cd','11')), axis=1).rename(columns={'위도':'하차위도','경도':'하차경도'})
    result = pd.concat([result, ride, goff], axis=1)
    output = DATA_DIR / f'gtx_a_transport_card_{date}_raw_with_coords.csv'
    result.to_csv(output, index=False, encoding='utf-8-sig')
    print(f'{date}: {output.name} 저장 ({len(result):,}건) / 승차 {result["승차위도"].notna().mean():.1%} / 하차 {result["하차위도"].notna().mean():.1%}')

## 3개 추가 노선 좌표 매핑

9030, 9030-1, M7111의 정류장명 매핑 결과에 국토부 좌표를 붙인다.

### 셀 2. 추가 3개 노선 정류장명에 국토부 위·경도 매핑\n

### ?? ? 2. ?? 3? ?? ???? ????? ?? ??


In [ ]:
# 3개 추가 노선 전용 셀: route_stops 폴더는 사용하지 않음
input_path = DATA_DIR / 'gtx_a_transport_card_20241017_raw_extra_3routes_with_stop_names.csv'
output_path = DATA_DIR / 'gtx_a_transport_card_20241017_raw_extra_3routes_with_coords.csv'
extra = pd.read_csv(input_path, dtype=str, encoding='utf-8-sig').fillna('')
ride = extra.apply(lambda r: coordinate_by_name(r['승차정류장명'], r.get('ride_ctpv_cd','41')), axis=1).rename(columns={'위도':'승차위도','경도':'승차경도'})
goff = extra.apply(lambda r: coordinate_by_name(r['하차정류장명'], r.get('goff_ctpv_cd','11')), axis=1).rename(columns={'위도':'하차위도','경도':'하차경도'})
extra_coords = pd.concat([extra, ride, goff], axis=1)
extra_coords.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'3개 노선 좌표 매핑 저장: {output_path} ({len(extra_coords):,}건)')
print(f'승차 좌표 매칭률: {extra_coords["승차위도"].notna().mean():.1%}')
print(f'하차 좌표 매칭률: {extra_coords["하차위도"].notna().mean():.1%}')